# Project 11: neemt dengue in Suriname toe na zware regen?

**De vraag.** Volgen er meer denguegevallen in de weken na zware regen in Suriname? Als de gegevens een verband laten zien, is dat verband dan sterker dan toeval alleen zou geven?

**Het publiek.** Een buurtcomité in Paramaribo dat schoonmaakacties tegen stilstaand water organiseert. Het comité krijgt een bericht van één pagina dat zegt wat de gegevens laten zien over regen en dengue, hoe zeker iemand daarvan kan zijn, en wat de Wereldgezondheidsorganisatie huishoudens aanraadt.

**De data.** Twee openbare tabellen, per week gekoppeld. Wekelijkse denguegevallen voor heel Suriname komen uit de OpenDengue-database (versie 1.3, opgebouwd uit de weekrapporten van PAHO). Dagelijkse regenval voor de modelcel die het dichtst bij Paramaribo ligt, komt uit het archief van Open-Meteo (de ERA5-heranalyse), opgeteld in dezelfde weken. Beide zijn gedownload op 2026-11-06.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, één grafiek, het bericht van één pagina voor het buurtcomité, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of bouwt hem uit de twee openbare bronnen als het bestand er niet is. Elke rij is één week, beginnend op zondag. `cases` is het aantal denguegevallen dat die week werd gemeld. `zero_filled` is True wanneer de database een nul invulde omdat er geen melding binnenkwam. `rain_mm` is de regen die die week viel, in millimeter.

In [ ]:
import io, json, zipfile, urllib.request
import numpy as np
import pandas as pd

DENGUE_URL = "https://opendengue.org/assets/Temporal_extract_PAHO_V1_3.zip"
RAIN_URL = ("https://archive-api.open-meteo.com/v1/archive?latitude=5.85&longitude=-55.2"
            "&start_date=2017-12-31&end_date=2024-12-28&daily=precipitation_sum"
            "&timezone=America%2FParamaribo&models=era5")
try:
    dengue = pd.read_csv("../data/week11-dengue-rain.csv", parse_dates=["week_start"])
except FileNotFoundError:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(DENGUE_URL).read()))
    d = pd.read_csv(z.open(z.namelist()[0]), low_memory=False)
    d = d[(d["adm_0_name"] == "SURINAME") & (d["T_res"] == "Week")]
    d = pd.DataFrame({"week_start": pd.to_datetime(d["calendar_start_date"]),
                      "cases": d["dengue_total"].astype(int),
                      "zero_filled": d["UUID"].str.contains("Zero filling")})
    d = d[(d["week_start"] >= "2017-12-31") & (d["week_start"] <= "2024-12-22")]
    daily = json.load(urllib.request.urlopen(RAIN_URL))["daily"]
    rain = pd.DataFrame({"day": pd.to_datetime(daily["time"]), "mm": daily["precipitation_sum"]})
    rain["week_start"] = rain["day"] - pd.to_timedelta((rain["day"].dt.dayofweek + 1) % 7, unit="D")
    rain = rain.groupby("week_start")["mm"].sum().round(1).rename("rain_mm").reset_index()
    dengue = d.merge(rain, on="week_start", how="left").sort_values("week_start").reset_index(drop=True)
dengue

## TODO 1: beschrijf de tabel

Druk het aantal rijen af, de eerste en laatste week, en het totaal van `cases`. Sla in `n_zero_filled` op in hoeveel weken de database een nul invulde.

Een ingevulde nul is een week die niemand meldde. Het is geen week zonder dengue.

In [ ]:
# TODO 1
n_zero_filled = None

In [ ]:
# CHECK 1
assert n_zero_filled is not None, "n_zero_filled is nog leeg"
assert n_zero_filled == int(dengue["zero_filled"].sum())
assert (dengue["week_start"].diff().dropna() == pd.Timedelta(weeks=1)).all()
print("CHECK 1 geslaagd:", n_zero_filled, "weken zijn met een nul gevuld")

## TODO 2: de regen vóór elke week

Regen veroorzaakt niet in dezelfde week een denguegeval. De Amerikaanse Centers for Disease Control and Prevention zeggen dat een muggenei 7 tot 10 dagen nodig heeft om een volwassen mug te worden, en de Wereldgezondheidsorganisatie zegt dat klachten 4 tot 10 dagen na de besmetting beginnen. Meldingen kosten nog meer tijd.

Het project legt zijn venster vast voordat het naar de resultaten kijkt: de totale regen in week 3 tot en met 6 vóór elke week. Voeg dat totaal toe als kolom `rain_before`. Bereken het op de volledige tabel, voordat je weken weglaat, zodat de regen van elke week op de juiste plek blijft.

In [ ]:
# TODO 2

In [ ]:
# CHECK 2
assert "rain_before" in dengue, "voeg de kolom rain_before toe"
dates = dengue.set_index("week_start")["rain_mm"]
for i in [6, 120, 300]:
    t = dengue["week_start"][i]
    expected = sum(dates[t - pd.Timedelta(weeks=k)] for k in range(3, 7))
    assert abs(dengue["rain_before"][i] - expected) < 0.05, f"rij {i} klopt niet"
assert dengue["rain_before"].isna().sum() == 6
print("CHECK 2 geslaagd: de week van", dengue["week_start"][120].date(), "had", round(dengue["rain_before"][120], 1), "mm in week 3 tot en met 6 ervoor")

## TODO 3: houd de bruikbare weken

Sla in `weeks` de rijen op die gemeld zijn (niet met een nul gevuld) en een waarde hebben voor `rain_before`.

In [ ]:
# TODO 3
weeks = None

In [ ]:
# CHECK 3
assert weeks is not None, "weeks is nog leeg"
assert not weeks["zero_filled"].any() and weeks["rain_before"].notna().all()
assert len(weeks) == int((~dengue["zero_filled"] & dengue["rain_before"].notna()).sum())
print("CHECK 3 geslaagd:", len(weeks), "weken om mee te werken")

## TODO 4: grafiek van gevallen en regen

Maak twee grafieken onder elkaar met dezelfde tijdas: denguegevallen per week en regen per week. Geef elke grafiek een titel en benoemde assen, en noem beide bronnen. Markeer waar de met nul gevulde weken zitten, of laat ze weg, en zeg wat je deed.

In [ ]:
# TODO 4

## TODO 5: het verschil

Noem een week **nat vooraf** als haar `rain_before` boven de mediaan van `rain_before` in `weeks` ligt. Sla die labels op als een True/False-array in `wet`. Sla in `gap` het gemiddelde aantal gevallen in natte weken op, min het gemiddelde in de andere weken.

In [ ]:
# TODO 5
wet = None
gap = None

In [ ]:
# CHECK 5
cases = weeks["cases"].to_numpy()
w = (weeks["rain_before"] > weeks["rain_before"].median()).to_numpy()
assert wet is not None and gap is not None, "wet of gap is nog leeg"
assert (np.asarray(wet) == w).all(), "wet moet True zijn boven de mediaan"
assert abs(gap - (cases[w].mean() - cases[~w].mean())) < 1e-9, "gap moet nat min de andere weken zijn"
print("CHECK 5 geslaagd: verschil van", round(gap, 2), "gevallen per week")

## TODO 6: gewoon schudden

Schud de labels in `wet` 10.000 keer met `np.random.default_rng(11)`, bereken elke keer het verschil opnieuw, en sla in `p_shuffle` het aandeel schudbeurten op met een verschil dat in beide richtingen minstens zo groot is als `gap`.

In [ ]:
# TODO 6
p_shuffle = None

In [ ]:
# CHECK 6
assert p_shuffle is not None, "p_shuffle is nog leeg"
checker = np.random.default_rng(5)
ref = np.array([(lambda s: cases[s].mean() - cases[~s].mean())(checker.permutation(w)) for _ in range(10000)])
p_ref = np.mean(np.abs(ref) >= abs(gap))
assert abs(p_shuffle - p_ref) < 0.005, "p_shuffle klopt niet: tel beide richtingen"
print("CHECK 6 geslaagd: p_shuffle dicht bij", round(p_ref, 4))

## TODO 7: de verschuivingstoets

Denguegevallen drijven van week tot week, en regen ook, dus gewoon schudden breekt de volgorde die in de data zit. Houd `cases` op zijn plek en schuif de labels in `wet` erlangs met `np.roll`, voor elke verschuiving van 26 weken tot `len(weeks) - 26` weken. Sla in `p_shift` het aandeel verschuivingen op met een verschil dat in beide richtingen minstens zo groot is als `gap`.

In [ ]:
# TODO 7
p_shift = None

In [ ]:
# CHECK 7
assert p_shift is not None, "p_shift is nog leeg"
shifted = np.array([(lambda s: cases[s].mean() - cases[~s].mean())(np.roll(w, k)) for k in range(26, len(w) - 25)])
assert abs(p_shift - np.mean(np.abs(shifted) >= abs(gap))) < 1e-9, "p_shift klopt niet: controleer het bereik van de verschuivingen"
print("CHECK 7 geslaagd:", len(shifted), "verschuivingen, p_shift", round(p_shift, 4))

## TODO 8: de AI-controle

Vraag je AI-assistent, zonder hem de data te laten zien: *"Nemen denguegevallen in Suriname toe na zware regen? Hoeveel weken later?"* Schrijf het antwoord en de vertraging die hij noemt in jullie AI-logboek.

Toets het daarna. Maak een tabel `lag_table` met één rij voor elke vertraging van 0 tot 12 weken en drie kolommen: `lag`, `gap` (het verschil als "nat" alleen de regen gebruikt van de ene week die zoveel weken eerder viel) en `p_shift` (de verschuivingstoets voor dat verschil). Gebruik dezelfde rijen als `weeks` en laat rijen weg waarvan de eerdere regen ontbreekt. Vergelijk de vertraging die de AI noemde met je tabel.

Tel tot slot je pogingen. Je toetste 13 vertragingen, dus de kleinste p-waarde in de tabel is een winnaar uit vele. Zeg in het logboek wat dat betekent voor elke vertraging die sterk lijkt.

In [ ]:
# TODO 8
lag_table = None

In [ ]:
# CHECK 8
assert lag_table is not None, "lag_table is nog leeg"
assert list(lag_table["lag"]) == list(range(13))
for k in [0, 4, 12]:
    lagged = dengue["rain_mm"].shift(k).loc[weeks.index]
    ok = lagged.notna().to_numpy()
    c, s = cases[ok], (lagged[ok] > lagged[ok].median()).to_numpy()
    assert abs(lag_table["gap"][k] - (c[s].mean() - c[~s].mean())) < 0.01, f"het verschil bij vertraging {k} klopt niet"
print("CHECK 8 geslaagd: vertragingen met meer gevallen na natte weken:", list(lag_table.loc[lag_table["gap"] > 0, "lag"]))

## TODO 9: het bericht voor het buurtcomité

Schrijf in een tekstcel hieronder een bericht van ongeveer 250 woorden voor het buurtcomité. Zeg wat de gegevens laten zien over regen en dengue, en zeg in woorden die een bewoner gebruikt hoe vaak toeval alleen dat zou opleveren. Noem beide bronnen. Zeg twee dingen die de data het comité niet kunnen vertellen, en herhaal het advies van de Wereldgezondheidsorganisatie over wateropslag, met de link.